In [14]:
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_16'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Working directory: /content/BITS_programming/assignments/assignment_16


# 🚀 WEEK 1 · Session 01: End-to-End MLOps & Production Readiness

**Programme:** BITS Pilani Professional AI/ML Programme  
**Level:** Intermediate &nbsp;|&nbsp; **Format:** Architect-led walkthrough + AWS hands-on  
**AWS Services:** Amazon S3 · SageMaker Studio · SageMaker Processing · Model Registry · CloudWatch · IAM

---

## 📖 How to Use This Notebook

| Mode | Setting | When to Use |
|------|---------|-------------|
| 📚 **Study Mode** | `RUN_AWS = False` *(default)* | Explore concepts, run simulations, take quizzes — **no AWS account needed** |
| 🔬 **Lab Mode** | `RUN_AWS = True` | Run inside an **approved BITS sandbox** — real resources, real costs |
| 🔁 **Review Mode** | Any | Re-read concept cells & quiz yourself after the live session |

> **💡 Tip:** Every AWS call in Study Mode prints a realistic simulation of what AWS would return. You can learn the full workflow without spending a rupee on cloud resources.

---

## 🗺️ Your Learning Journey

```
┌──────────────────────────────────────────────────────────────┐
│  SECTION 1  → The Production Gap (concept + analogy)        │
│  SECTION 2  → MLOps vs DevOps + Maturity Levels (quiz)      │
│  SECTION 3  → AWS Architecture (visual overview)            │
│  SECTION 4  → Implementation A: Build a production baseline │
│  SECTION 5  → Implementation B: Register & govern a model   │
│  SECTION 6  → Practice Zone (local simulation — no AWS!)    │
│  SECTION 7  → Final Quiz + Summary                          │
└──────────────────────────────────────────────────────────────┘
```


---
## 📌 SECTION 1 — The Production Gap: Why Notebooks Aren't Enough

### 🍜 The Restaurant Analogy

Imagine you are a **home cook** who makes amazing biryani in your kitchen.  
Everything works perfectly — your ingredients, your stove, your taste.

Now a restaurant chain says: *"Can you produce 10,000 servings a day, across 50 cities, with consistent quality and 24/7 availability?"*

Suddenly, your recipe needs:

| Home Kitchen | Restaurant Kitchen |
|---|---|
| One cook, one recipe | Standardized procedures, multiple chefs |
| Cook whenever you want | Scheduled shifts, SLAs |
| Taste-test yourself | Quality control team |
| Buy ingredients day-of | Supply chain management |
| No audit trail | Compliance logs for every batch |

**Machine learning is exactly the same.**

---

### 🤖 From Notebook to Production — What Actually Changes?

```
NOTEBOOK WORLD                       PRODUCTION WORLD
──────────────────────────────────────────────────────────
✓ You control the data               ✗ Data arrives from multiple sources
✓ You know the environment           ✗ 10+ engineers share infra
✓ You decide when to retrain         ✗ Automated triggers based on drift
✓ Bugs affect only you               ✗ Bugs affect 100k users
✓ "It runs on my machine"            ✗ Runs on Docker, Kubernetes, cloud
✓ Accuracy tracked in a notebook     ✗ Accuracy tracked in dashboards 24/7
✓ You fix issues manually            ✗ Auto-rollback when health checks fail
```

**Key insight:** Production ML is not just "running the model" — it's a **sociotechnical system** involving people, processes, and infrastructure.


### 💥 Real Production Failures — Lessons from Industry

These are **real** incidents where the gap between notebooks and production caused problems:

---

**🔴 Example 1: Amazon Hiring Algorithm (2018)**  
An ML model trained on 10 years of hiring data learned to *penalize resumes mentioning "women's"* (as in "women's chess club").  
**Why it happened:** Training data reflected historical bias. No fairness audit before deployment.  
**Production lesson:** Models need **data validation + fairness checks** before going live.

---

**🔴 Example 2: Tesla Autopilot (2016)**  
An Autopilot model failed to distinguish a white truck against a bright sky. Tragic accident.  
**Why it happened:** Edge case not in training data. No real-world adversarial testing.  
**Production lesson:** **Out-of-distribution detection** and safety gates are non-negotiable.

---

**🔴 Example 3: Zillow iBuying Algorithm (2021)**  
Zillow's home price prediction model miscalculated, leading to $569M in losses. They bought homes at prices higher than they could sell.  
**Why it happened:** Model drift during COVID disrupted patterns. No automated drift monitoring.  
**Production lesson:** **Model monitoring** (detecting when the world changes) is critical.

---

> 🎯 **Pattern:** Every failure traces back to gaps in validation, monitoring, or governance — the exact things MLOps addresses.


---
## 📌 SECTION 2 — MLOps vs DevOps + Maturity Levels

### 🔄 MLOps vs DevOps: Same Family, Different Challenges

Think of DevOps and MLOps like **regular construction vs. building a living house**.

| Dimension | DevOps (Software) | MLOps (ML Systems) |
|-----------|-------------------|--------------------|
| **Code** | Deterministic — same input, same output | Probabilistic — output depends on data |
| **Testing** | Unit tests, integration tests | Accuracy metrics, bias tests, drift detection |
| **Versioning** | Git for code | Git + DVC for **code + data + models** |
| **Deployment** | Deploy new version of app | Deploy new version of **model + serving infra** |
| **Rollback** | Re-deploy old code | Re-deploy old model + validate old predictions |
| **Monitoring** | CPU, memory, latency | **Plus:** accuracy, data drift, feature drift |
| **Failure mode** | App crashes → obvious | Model degrades silently → insidious |

> **Key difference:** ML systems have a third artifact — **the model** — that can go stale without the code changing at all. This is the core challenge MLOps solves.

---

### 📈 MLOps Maturity Levels: Where Is Your Team?

Think of these like **belt levels in martial arts**:

```
Level 0 (White Belt) — Manual, ad-hoc
  └── Data scientists work in notebooks
  └── Model is a .pkl file emailed to the DevOps team
  └── Deployment is "copy this file to the server"
  └── ⚠️ Most Indian enterprise teams are here today

Level 1 (Yellow Belt) — Scripted, repeatable  
  └── Training is a Python script, not just a notebook
  └── Model goes to a model registry (not email!)
  └── Basic CI/CD pipeline exists
  └── Monitoring: someone checks accuracy monthly

Level 2 (Green Belt) — Automated training
  └── Continuous Training (CT) pipeline runs on new data automatically
  └── Model evaluation is automated with thresholds
  └── A/B testing before full rollout
  └── Monitoring: automated alerts on drift

Level 3 (Blue Belt) — Automated deployment
  └── Entire CT/CD pipeline is automated end-to-end
  └── New model deploys automatically if it passes gates
  └── Feature store ensures training-serving consistency
  └── Monitoring: real-time dashboards, auto-rollback

Level 4 (Black Belt) — Self-healing systems
  └── Models retrain themselves when drift is detected
  └── Automated root cause analysis
  └── Chaos engineering for ML systems
  └── ⚠️ Google, Meta, Netflix operate here
```


In [15]:
# ============================================================
# 🧠 SELF-ASSESSMENT: What Maturity Level Are You At?
# ============================================================
#
# For each question, score:
#   0 = We don't do this
#   1 = We do this manually
#   2 = We do this with scripts
#   3 = This is fully automated
#
# Change the scores below based on your current team's practice:

assessment = {
    "Model training is reproducible (same data → same model)":        1,  # ← Change 0-3
    "Model versions are tracked with metadata":                        1,  # ← Change 0-3
    "Training happens automatically when new data arrives":            0,  # ← Change 0-3
    "Model performance is monitored in production":                    1,  # ← Change 0-3
    "Deployment is automated (no manual steps)":                       0,  # ← Change 0-3
    "Data quality checks run before training":                         1,  # ← Change 0-3
    "We can roll back a model in under 10 minutes":                   1,  # ← Change 0-3
    "Feature engineering is shared between training and serving":      0,  # ← Change 0-3
}

# ─── Score calculation ───────────────────────────────────────
total = sum(assessment.values())
max_score = len(assessment) * 3
pct = (total / max_score) * 100

if pct < 25:
    level, advice = "Level 0 — Manual", "Focus on making training reproducible first."
elif pct < 50:
    level, advice = "Level 1 — Scripted", "Add a model registry and basic CI pipeline next."
elif pct < 70:
    level, advice = "Level 2 — Automated Training", "Invest in monitoring and automated deployment gates."
elif pct < 90:
    level, advice = "Level 3 — Automated Deployment", "Work on feature stores and self-healing patterns."
else:
    level, advice = "Level 4 — Self-Healing", "You're world-class! Focus on chaos engineering."

print("=" * 60)
print("  MLOps Maturity Self-Assessment")
print("=" * 60)
for question, score in assessment.items():
    bar = "█" * score + "░" * (3 - score)
    print(f"  [{bar}] {score}/3  {question[:48]}...")
print()
print(f"  Total Score : {total}/{max_score}  ({pct:.0f}%)")
print(f"  Your Level  : {level}")
print(f"  Next Step   : {advice}")
print("=" * 60)
print()
print("💡 This course targets Level 1→3 transitions using AWS.")


  MLOps Maturity Self-Assessment
  [█░░] 1/3  Model training is reproducible (same data → same...
  [█░░] 1/3  Model versions are tracked with metadata...
  [░░░] 0/3  Training happens automatically when new data arr...
  [█░░] 1/3  Model performance is monitored in production...
  [░░░] 0/3  Deployment is automated (no manual steps)...
  [█░░] 1/3  Data quality checks run before training...
  [█░░] 1/3  We can roll back a model in under 10 minutes...
  [░░░] 0/3  Feature engineering is shared between training a...

  Total Score : 5/24  (21%)
  Your Level  : Level 0 — Manual
  Next Step   : Focus on making training reproducible first.

💡 This course targets Level 1→3 transitions using AWS.


### ⚙️ The Three Pillars of Production ML

Any production ML system must satisfy three properties. Miss one and you have a problem.

---

**🔷 1. Reliability**  
> *"The system does what it's supposed to do, even when things go wrong."*

Examples of reliability requirements:
- Model serves predictions even if one data source goes down (fallback logic)
- If the model fails, there's a graceful degradation (serve cached result or default)
- SLA: 99.9% uptime = max 8.7 hours downtime per year

---

**🔷 2. Scalability**  
> *"The system handles 10x traffic without code changes."*

Examples of scalability requirements:
- During Diwali sales, prediction traffic spikes 50x — auto-scaling handles it
- Model training on 1TB of data runs in the same time as on 10GB (distributed training)
- Feature computation scales horizontally with more workers

---

**🔷 3. Security**  
> *"Only the right people and systems can access what they need."*

Examples of security requirements:
- Model weights are encrypted at rest in S3
- Only the inference service role can read the model artifact (IAM least-privilege)
- All API calls are logged in CloudTrail for audit
- No personally identifiable information (PII) leaks through model outputs

---

> 🎯 **AWS provides all three:** S3 + IAM for security, SageMaker auto-scaling for scalability, CloudWatch alerts for reliability.


---
## 📌 SECTION 3 — AWS Architecture for This Session

### 🏗️ What We're Building

Here's the architecture we'll implement. Study it before running any code.

```
                        ┌─────────────────────────────────────────────┐
                        │           AWS Account (Sandbox)             │
                        │                                             │
    ┌──────────┐         │   ┌─────────────┐      ┌──────────────┐   │
    │          │  IAM    │   │  SageMaker  │      │  SageMaker   │   │
    │ Notebook │ ──────▶ │   │   Studio    │ ───▶ │  Processing  │   │
    │  (You)   │ Role    │   │ (IDE + Run) │      │    Job       │   │
    │          │         │   └─────────────┘      └──────┬───────┘   │
    └──────────┘         │                               │           │
                        │                               │ outputs    │
                        │   ┌─────────────┐      ┌──────▼───────┐   │
                        │   │  SageMaker  │      │   Amazon S3  │   │
                        │   │   Model     │ ◀─── │  (Evidence   │   │
                        │   │  Registry   │      │   Storage)   │   │
                        │   └──────┬──────┘      └──────────────┘   │
                        │          │                                  │
                        │   ┌──────▼──────┐                          │
                        │   │  Amazon     │                          │
                        │   │  CloudWatch │ ◀── Logs + Metrics       │
                        │   │  (Monitor)  │                          │
                        │   └─────────────┘                          │
                        └─────────────────────────────────────────────┘
```

### 🧩 What Each Service Does in Our Session

| Service | Role in This Session | Analogy |
|---------|---------------------|---------|
| **AWS IAM** | Controls *who* can do *what* | Security guard at building entrance |
| **Amazon S3** | Stores all our evidence, manifests, and artifacts | Filing cabinet |
| **SageMaker Studio** | Our cloud IDE (where we run the notebook) | Your office desk in the cloud |
| **SageMaker Processing** | Runs data processing jobs at scale | Dedicated processing department |
| **SageMaker Model Registry** | Catalogs approved models with metadata | Library card catalog for models |
| **Amazon CloudWatch** | Monitors logs, metrics, and sends alerts | Security camera + alarm system |


In [16]:
# ============================================================
# ⚙️ CELL: Core Setup — Run This First!
# ============================================================
#
# This cell sets up everything you need for both Study and Lab modes.
# Key decisions made here:
#
#   RUN_AWS = False  →  All AWS calls are intercepted by MockAWS
#                       and print realistic simulated responses
#
#   RUN_AWS = True   →  Calls go to real AWS (requires sandbox access)

RUN_AWS    = False          # ← LEAVE AS FALSE until instructor says otherwise
AWS_REGION = 'us-east-2'   # ← AWS region for all resources
PROJECT    = 'bits-ai-platform-engineering'
SESSION_ID = 'session_01'

# ─── Standard library imports ───────────────────────────────
import json, time, os, datetime, uuid, textwrap, pprint

# ─── Study-mode simulator ───────────────────────────────────
class MockAWS:
    """Simulates AWS API responses so you can learn the workflow
    without cloud access. In production, these calls hit real AWS."""

    @staticmethod
    def put_object(bucket, key, body):
        print(f"  📦 S3 PUT  → s3://{bucket}/{key}")
        print(f"     Size   : {len(body)} bytes")
        return {'ETag': f'"{uuid.uuid4().hex[:16]}"'}

    @staticmethod
    def create_processing_job(name, instance_type='ml.m5.large'):
        print(f"  ⚙️  SageMaker Processing Job created:")
        print(f"     Name    : {name}")
        print(f"     Instance: {instance_type}")
        print(f"     Status  : InProgress → Completed (in ~8 min live)")
        return {'ProcessingJobArn': f'arn:aws:sagemaker:{AWS_REGION}:123456:processing-job/{name}'}

    @staticmethod
    def create_model_package_group(group_name):
        print(f"  📚 Model Registry group created: {group_name}")
        return {'ModelPackageGroupArn': f'arn:aws:sagemaker:{AWS_REGION}:123456:model-package-group/{group_name}'}

def guard():
    """Call this before any real AWS resource creation."""
    if not RUN_AWS:
        raise RuntimeError('RUN_AWS=False. Review costs and permissions, then set True.')

# ─── Run suffix for unique resource names ───────────────────
run_suffix = datetime.datetime.utcnow().strftime('%Y%m%d%H%M%S')

# ─── Status banner ──────────────────────────────────────────
mode_label  = '🔴 LIVE AWS — real resources will be created!' if RUN_AWS else '🟢 STUDY MODE — safe simulation'
print("=" * 60)
print(f"  {'Session':<12}: {SESSION_ID}")
print(f"  {'Project':<12}: {PROJECT}")
print(f"  {'Region':<12}: {AWS_REGION}")
print(f"  {'Run ID':<12}: {run_suffix}")
print(f"  {'Mode':<12}: {mode_label}")
print("=" * 60)
if not RUN_AWS:
    print()
    print("  Everything below will SIMULATE what AWS would do.")
    print("  You'll see the exact same JSON structures as live AWS.")


  Session     : session_01
  Project     : bits-ai-platform-engineering
  Region      : us-east-2
  Run ID      : 20260910070242
  Mode        : 🟢 STUDY MODE — safe simulation

  Everything below will SIMULATE what AWS would do.
  You'll see the exact same JSON structures as live AWS.


/tmp/ipykernel_2337/2603949660.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_suffix = datetime.datetime.utcnow().strftime('%Y%m%d%H%M%S')


In [17]:
# ============================================================
# 📦 CELL: AWS SDK Setup
# ============================================================
#
# boto3 is the official Python SDK for AWS.
# Think of it like 'requests' but for AWS services.
#
# In SageMaker Studio, boto3 is pre-installed.
# For local testing: pip install boto3

try:
    import boto3
    from botocore.exceptions import ClientError
    print("✅ boto3 imported successfully")
    print(f"   Version: {boto3.__version__}")
except ImportError:
    print("⚠️  boto3 not found. In SageMaker Studio this is pre-installed.")
    print("   For local install: pip install boto3")
    # Create a minimal stub so cells below don't crash in study mode
    class boto3:
        @staticmethod
        def Session(**kwargs): return None

# ─── Create session ─────────────────────────────────────────
# A boto3 Session holds your credentials and region.
# Think of it as 'logging in' to AWS programmatically.
if RUN_AWS:
    session    = boto3.Session(region_name=AWS_REGION)
    sts        = session.client('sts')                          # Security Token Service
    account_id = sts.get_caller_identity()['Account']
    s3         = session.client('s3')
    sm         = session.client('sagemaker')
    print(f"✅ Connected to AWS account: {account_id}")
else:
    session    = None
    account_id = '111122223333'                                 # Fake account ID for simulation
    s3         = None
    sm         = None
    print(f"🟢 Study mode — using simulated account ID: {account_id}")

print()
print("🔑 IAM Role pattern used by SageMaker:")
role_arn = f'arn:aws:iam::{account_id}:role/SageMakerExecutionRole'
print(f"   {role_arn}")
print()
print("💡 This IAM role is what SageMaker 'becomes' when it runs jobs.")
print("   It must have permissions to: read S3, write S3, create jobs.")


⚠️  boto3 not found. In SageMaker Studio this is pre-installed.
   For local install: pip install boto3
🟢 Study mode — using simulated account ID: 111122223333

🔑 IAM Role pattern used by SageMaker:
   arn:aws:iam::111122223333:role/SageMakerExecutionRole

💡 This IAM role is what SageMaker 'becomes' when it runs jobs.
   It must have permissions to: read S3, write S3, create jobs.


In [18]:
# ============================================================
# 🗂️ CELL: S3 Workspace Setup
# ============================================================
#
# Why do we need a structured S3 workspace?
#
# S3 is the "filing cabinet" of our MLOps system. We use it to:
#   1. Store training data (input)
#   2. Store model artifacts (output)
#   3. Store audit evidence (compliance)
#   4. Share data between processing steps (intermediate)
#
# Best practice: use a consistent path structure so any team member
# can find any artifact without asking you.

# ─── Naming convention ──────────────────────────────────────
# We include account_id + region to make bucket names globally unique
# (S3 bucket names must be unique ACROSS ALL AWS customers worldwide!)
bucket = f'{PROJECT}-{account_id}-{AWS_REGION}'.replace('_','-')
prefix = f'{SESSION_ID}/runs/{run_suffix}'

print("S3 Workspace Structure:")
print(f"  Bucket : s3://{bucket}/")
print(f"  ├── {prefix}/")
print(f"  │   ├── manifest.json            ← What we built and why")
print(f"  │   ├── processing/              ← Processing job outputs")
print(f"  │   ├── release/evidence.json    ← Pre-release gate record")
print(f"  │   └── cleanup/checklist.json  ← Verified cleanup record")
print()

# ─── Resource tagging ───────────────────────────────────────
# Tags are KEY for cost management and governance.
# At a large company, these tags determine your team's AWS bill.
tags = [
    {'Key': 'project', 'Value': PROJECT},
    {'Key': 'session', 'Value': SESSION_ID},
    {'Key': 'owner',   'Value': 'bits-platform-engineering'},
    {'Key': 'expires', 'Value': (datetime.datetime.utcnow() + datetime.timedelta(days=2)).strftime('%Y-%m-%d')},
]
print("Resource Tags (applied to ALL resources we create):")
for tag in tags:
    print(f"  {tag['Key']:<10} = {tag['Value']}")
print()
print("💡 'expires' tag tells the ops team when to delete resources.")
print("   Without it, forgotten resources = unexpected costs! 💸")

# ─── Create bucket (only in live mode) ──────────────────────
if RUN_AWS:
    try:
        s3.head_bucket(Bucket=bucket)
        print(f"\n✅ Bucket already exists: {bucket}")
    except ClientError:
        kwargs = {'Bucket': bucket}
        if AWS_REGION == 'us-east-2':
            kwargs['CreateBucketConfiguration'] = {'LocationConstraint': AWS_REGION}
        s3.create_bucket(**kwargs)
        print(f"\n✅ Created bucket: {bucket}")
else:
    print(f"\n🔵 [SIMULATED] Would create S3 bucket: {bucket}")


S3 Workspace Structure:
  Bucket : s3://bits-ai-platform-engineering-111122223333-us-east-2/
  ├── session_01/runs/20260910070242/
  │   ├── manifest.json            ← What we built and why
  │   ├── processing/              ← Processing job outputs
  │   ├── release/evidence.json    ← Pre-release gate record
  │   └── cleanup/checklist.json  ← Verified cleanup record

Resource Tags (applied to ALL resources we create):
  project    = bits-ai-platform-engineering
  session    = session_01
  owner      = bits-platform-engineering
  expires    = 2026-09-12

💡 'expires' tag tells the ops team when to delete resources.
   Without it, forgotten resources = unexpected costs! 💸

🔵 [SIMULATED] Would create S3 bucket: bits-ai-platform-engineering-111122223333-us-east-2


/tmp/ipykernel_2337/1875784799.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {'Key': 'expires', 'Value': (datetime.datetime.utcnow() + datetime.timedelta(days=2)).strftime('%Y-%m-%d')},


---
## 📌 SECTION 4 — Implementation A: Build a Production Baseline

### 🎯 What We're Doing Here

In Implementation A, we build the **foundations of a production-grade ML run**:

1. **Write a manifest** → An auditable record of *what* we're building and *why*
2. **Create a processing job** → Run data validation/preparation at scale on SageMaker
3. **Register a model package group** → A named "drawer" in SageMaker Model Registry

### 🤔 Why Start with a Manifest?

Think of a manifest as the **"intent declaration"** before you touch any cloud resource.

> Real-world parallel: Before any construction starts, a site engineer files a blueprint + permit.  
> The manifest is our blueprint + permit.

A manifest answers:
- *When* did this run happen? (for debugging)
- *What* services and data did it use? (for auditing)
- *Who* is responsible? (for accountability)
- *What controls* are in place? (for governance)

Without a manifest, when something breaks at 3 AM six months later, your team is guessing. With one, they trace it in minutes.


In [19]:
# ============================================================
# 📋 CELL: Implementation A — Write the Run Manifest
# ============================================================
#
# We create this BEFORE creating any resources.
# This way, even if the run fails halfway, we have a record of intent.

manifest = {
    'project'      : PROJECT,
    'session_id'   : SESSION_ID,
    'aws_region'   : AWS_REGION,
    'services'     : [
        'Amazon S3',
        'Amazon SageMaker Studio',
        'SageMaker Processing',
        'SageMaker Model Registry',
        'Amazon CloudWatch',
        'AWS IAM',
    ],
    'created_at_utc': datetime.datetime.utcnow().isoformat(),
    'run_id'       : str(uuid.uuid4()),
    'controls'     : [
        'versioned S3 evidence',        # Every artifact has a unique path
        'least-privilege IAM review',   # Role can only do what it needs
        'CloudWatch observability',     # Logs and metrics captured
        'explicit cleanup',             # Resources are deleted after session
    ],
    # ─── New for learners: describe what this run is testing ───
    'learning_objectives': [
        'Understand production manifest structure',
        'Practice IAM role scoping',
        'Observe SageMaker Processing job lifecycle',
        'Register a model package group',
    ]
}

print("Run Manifest (will be written to S3):")
print(json.dumps(manifest, indent=2))
print()

# ─── Write manifest to S3 ───────────────────────────────────
manifest_key  = f'{prefix}/manifest.json'
manifest_body = json.dumps(manifest, indent=2).encode('utf-8')

if RUN_AWS:
    s3.put_object(Bucket=bucket, Key=manifest_key, Body=manifest_body)
    print(f"✅ Manifest written to: s3://{bucket}/{manifest_key}")
else:
    MockAWS.put_object(bucket, manifest_key, manifest_body)

print()
print("📌 Key fields to remember:")
print("   run_id    → unique per execution (use to find logs in CloudWatch)")
print("   controls  → governance checklist items")
print("   services  → what we have to clean up at the end")


Run Manifest (will be written to S3):
{
  "project": "bits-ai-platform-engineering",
  "session_id": "session_01",
  "aws_region": "us-east-2",
  "services": [
    "Amazon S3",
    "Amazon SageMaker Studio",
    "SageMaker Processing",
    "SageMaker Model Registry",
    "Amazon CloudWatch",
    "AWS IAM"
  ],
  "created_at_utc": "2026-09-10T07:02:42.557802",
  "run_id": "86634a94-ad53-473d-9c43-3d408143963e",
  "controls": [
    "versioned S3 evidence",
    "least-privilege IAM review",
    "CloudWatch observability",
    "explicit cleanup"
  ],
  "learning_objectives": [
    "Understand production manifest structure",
    "Practice IAM role scoping",
    "Observe SageMaker Processing job lifecycle",
    "Register a model package group"
  ]
}

  📦 S3 PUT  → s3://bits-ai-platform-engineering-111122223333-us-east-2/session_01/runs/20260910070242/manifest.json
     Size   : 715 bytes

📌 Key fields to remember:
   run_id    → unique per execution (use to find logs in CloudWatch)
   contro

/tmp/ipykernel_2337/1636163475.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at_utc': datetime.datetime.utcnow().isoformat(),


### 🔬 Understanding SageMaker Processing Jobs

Before we create the processing job, let's understand **why** we use one.

#### The Problem with Running Data Prep in Your Notebook

| Notebook (BAD for production) | SageMaker Processing (GOOD) |
|-------------------------------|------------------------------|
| Runs on your laptop/Studio instance | Runs on **dedicated, isolated compute** |
| Fails if you close your laptop | Managed — survives network interruptions |
| Hard to reproduce exact environment | Docker container = reproducible environment |
| No audit trail | Job name, logs, inputs/outputs all recorded |
| Can't scale to 100GB datasets | Scale up instance type with one config change |

#### What a Processing Job Looks Like

```
Your Script  →  Docker Container  →  ml.m5.large instance  →  S3 outputs
     ↑               ↑                       ↑                    ↑
  data_prep.py   sklearn:latest           1 vCPU, 8GB        evidence/
```

#### Key Configuration Fields

- **ImageUri** — The Docker image that runs your script (e.g., AWS's sklearn container)
- **InstanceType** — `ml.m5.large` = 2 vCPU, 8GB RAM (~$0.115/hour)
- **S3Output** — Where results are uploaded when the job finishes
- **StoppingCondition** — Auto-stop after N seconds (cost protection!)


In [20]:
# ============================================================
# ⚙️ CELL: Implementation A — SageMaker Processing Job + Registry
# ============================================================

processing_job_name = f'{SESSION_ID}-processing-{run_suffix}'
registry_group      = f'{PROJECT}-{SESSION_ID}-registry'

# ─── Build the processing job configuration ─────────────────
# This is the full spec sent to AWS. Each field matters:
processing_request = {
    'ProcessingJobName': processing_job_name,

    # IAM role SageMaker assumes when running this job
    'RoleArn': role_arn,

    # Docker image + optional entrypoint
    # AWS provides pre-built images for sklearn, pytorch, tensorflow, etc.
    'AppSpecification': {
        'ImageUri': f'{account_id}.dkr.ecr.{AWS_REGION}.amazonaws.com/sklearn-processing:latest'
    },

    # Compute: what machine to run on
    # InstanceType options: ml.t3.medium (cheap), ml.m5.large (balanced), ml.c5.4xlarge (CPU-heavy)
    'ProcessingResources': {
        'ClusterConfig': {
            'InstanceCount' : 1,           # Number of parallel instances
            'InstanceType'  : 'ml.m5.large',
            'VolumeSizeInGB': 30           # Disk space for the job
        }
    },

    # Where outputs go after the job completes
    'ProcessingOutputConfig': {
        'Outputs': [{
            'OutputName': 'evidence',
            'S3Output': {
                'S3Uri'      : f's3://{bucket}/{prefix}/processing/',
                'LocalPath'  : '/opt/ml/processing/output',
                'S3UploadMode': 'EndOfJob'   # Upload all at once when done
            }
        }]
    },

    # Cost protection: auto-stop after 30 minutes
    'StoppingCondition': {'MaxRuntimeInSeconds': 1800},
    'Tags': tags
}

print("Processing Job Config:")
print(json.dumps(processing_request, indent=2)[:1200])
print("  ... [truncated] ...")
print()

# ─── Submit job to SageMaker ────────────────────────────────
if RUN_AWS:
    sm.create_processing_job(**processing_request)
    print(f"✅ Processing job submitted: {processing_job_name}")
    print("   Monitor at: https://console.aws.amazon.com/sagemaker/home#/processing-jobs")
else:
    MockAWS.create_processing_job(processing_job_name)

print()

# ─── Create Model Registry group ────────────────────────────
# A ModelPackageGroup is like a "folder" in the model registry.
# All versions of a model go into one group.
# Example: 'churn-prediction-v1', 'churn-prediction-v2' both live in 'churn-prediction'
print("Creating Model Registry Package Group:")
print(f"  Group Name: {registry_group}")
print()
if RUN_AWS:
    try:
        sm.create_model_package_group(
            ModelPackageGroupName=registry_group,
            ModelPackageGroupDescription='Governed package group for session implementation'
        )
        print(f"✅ Registry group created: {registry_group}")
    except ClientError as e:
        if e.response['Error']['Code'] != 'ValidationException': raise
        print(f"ℹ️  Registry group already exists (that's fine!)")
else:
    MockAWS.create_model_package_group(registry_group)

print()
print("📌 Summary of what we created:")
print(f"   S3 Manifest     → s3://{bucket}/{prefix}/manifest.json")
print(f"   Processing Job  → {processing_job_name}")
print(f"   Model Registry  → {registry_group}")


Processing Job Config:
{
  "ProcessingJobName": "session_01-processing-20260910070242",
  "RoleArn": "arn:aws:iam::111122223333:role/SageMakerExecutionRole",
  "AppSpecification": {
    "ImageUri": "111122223333.dkr.ecr.us-east-2.amazonaws.com/sklearn-processing:latest"
  },
  "ProcessingResources": {
    "ClusterConfig": {
      "InstanceCount": 1,
      "InstanceType": "ml.m5.large",
      "VolumeSizeInGB": 30
    }
  },
  "ProcessingOutputConfig": {
    "Outputs": [
      {
        "OutputName": "evidence",
        "S3Output": {
          "S3Uri": "s3://bits-ai-platform-engineering-111122223333-us-east-2/session_01/runs/20260910070242/processing/",
          "LocalPath": "/opt/ml/processing/output",
          "S3UploadMode": "EndOfJob"
        }
      }
    ]
  },
  "StoppingCondition": {
    "MaxRuntimeInSeconds": 1800
  },
  "Tags": [
    {
      "Key": "project",
      "Value": "bits-ai-platform-engineering"
    },
    {
      "Key": "session",
      "Value": "session_01"
    },


In [21]:
# ============================================================
# 🧠 KNOWLEDGE CHECK 1 — Answer these before moving on!
# ============================================================

def check(q_num, your_answer):
    answers = {
        1: {
            'correct': 'b',
            'explanation': (
                'Correct! The StoppingCondition with MaxRuntimeInSeconds prevents '
                'runaway jobs from costing thousands of rupees. Without it, a '
                'bug could run a job for days.'
            )
        },
        2: {
            'correct': 'c',
            'explanation': (
                'Correct! S3 bucket names must be globally unique across ALL AWS '
                'customers. That\'s why we include account_id + region — it '
                'virtually guarantees uniqueness.'
            )
        },
        3: {
            'correct': 'd',
            'explanation': (
                'Correct! The manifest is written BEFORE resource creation. '
                'This way, even if the job fails, there\'s an audit record '
                'of the intended run.'
            )
        },
    }
    a = answers[q_num]
    if your_answer.lower().strip() == a['correct']:
        print(f"  Q{q_num}: ✅ Correct! {a['explanation']}")
    else:
        print(f"  Q{q_num}: ❌ Not quite. Answer is '{a['correct'].upper()}'.")
        print(f"          {a['explanation']}")
    print()

print("=" * 60)
print("  KNOWLEDGE CHECK 1")
print("=" * 60)
print()
print("Q1: Why does the processing job have a StoppingCondition?")
print("    a) It's required by AWS SageMaker API")
print("    b) It prevents runaway costs if the job hangs")
print("    c) It ensures the job runs at least 30 minutes")
print("    d) It controls the number of output files")
print()
print("Q2: Why does the S3 bucket name include account_id and region?")
print("    a) It's required for IAM permissions to work")
print("    b) It helps CloudWatch find the logs")
print("    c) S3 bucket names must be globally unique across all of AWS")
print("    d) To make the bucket name easier to remember")
print()
print("Q3: When is the manifest written relative to resource creation?")
print("    a) After all resources are created successfully")
print("    b) Only if the processing job completes successfully")
print("    c) In parallel with resource creation")
print("    d) Before any resources are created")
print()
print("─" * 60)
print("Set your answers below and run this cell again:")
print()

YOUR_ANSWER_Q1 = "b"   # ← change to a, b, c, or d
YOUR_ANSWER_Q2 = "c"   # ← change to a, b, c, or d
YOUR_ANSWER_Q3 = "d"   # ← change to a, b, c, or d

check(1, YOUR_ANSWER_Q1)
check(2, YOUR_ANSWER_Q2)
check(3, YOUR_ANSWER_Q3)


  KNOWLEDGE CHECK 1

Q1: Why does the processing job have a StoppingCondition?
    a) It's required by AWS SageMaker API
    b) It prevents runaway costs if the job hangs
    c) It ensures the job runs at least 30 minutes
    d) It controls the number of output files

Q2: Why does the S3 bucket name include account_id and region?
    a) It's required for IAM permissions to work
    b) It helps CloudWatch find the logs
    c) S3 bucket names must be globally unique across all of AWS
    d) To make the bucket name easier to remember

Q3: When is the manifest written relative to resource creation?
    a) After all resources are created successfully
    b) Only if the processing job completes successfully
    c) In parallel with resource creation
    d) Before any resources are created

────────────────────────────────────────────────────────────
Set your answers below and run this cell again:

  Q1: ✅ Correct! Correct! The StoppingCondition with MaxRuntimeInSeconds prevents runaway jobs f

---
## 📌 SECTION 5 — Implementation B: Register and Deploy a Governed Model

### 🎯 What We're Doing Here

In Part B we move from "it runs" to "it's ready for production":

1. **Create an evidence bundle** → A structured record that proves the model meets production standards
2. **Apply release gates** → Checklist items that must pass before a model is "approved"
3. **Write a cleanup checklist** → Documented steps to remove resources when done

### 🔑 The Release Gate Concept

Think of release gates like **airport security checkpoints**:

```
Model Training  →  [Contract Gate]  →  [Accuracy Gate]  →  [Security Gate]  →  Production
                    Data schema         F1 > 0.85           IAM scoped
                    matches serving     No bias detected     No PII leaked
```

If the model fails ANY gate, it goes back. No exceptions.

This is what separates professional MLOps from "push and pray".

### 📋 Evidence Bundle Structure

The evidence bundle answers: *"How do I prove this model is safe to deploy?"*

| Evidence Item | What It Proves |
|---------------|----------------|
| `artifact_uri` | Where the model lives (so ops can find it) |
| `schema reviewed` | Input/output contract is documented |
| `logs available` | We can debug if something goes wrong |
| `failure path defined` | We know what happens on error |
| `rollback documented` | We can undo the deployment |
| `cleanup listed` | We won't waste cloud budget |


In [22]:
# ============================================================
# 📦 CELL: Implementation B — Operational Evidence Bundle
# ============================================================
#
# The evidence bundle is written to S3 as a JSON document.
# In a real MLOps system, this would be generated automatically
# by your CI/CD pipeline and attached to the model registry entry.

evidence = {
    # Where all artifacts for this run live
    'artifact_uri'   : f's3://{bucket}/{prefix}/',

    # These are the "gate checks" that must pass before approval.
    # In a mature system, these would be auto-populated by tests.
    'checks': [
        'schema or contract reviewed',      # Input/output spec matches training
        'logs available',                   # CloudWatch logs confirmed
        'failure path defined',             # What happens if model throws exception?
        'rollback or recovery path documented', # How do we undo this deployment?
        'cost cleanup listed',              # Resources are tagged + cleanup plan ready
    ],

    # Who is accountable for this model
    'service_owners': {
        'technical': 'instructor',          # Who to call when it breaks
        'business' : 'programme owner',    # Who approves business logic
    },

    # Timestamps for audit trail
    'created_at_utc' : datetime.datetime.utcnow().isoformat(),
    'run_id'         : manifest['run_id'],  # Links back to the manifest

    # Must be True before we're done
    'cleanup_required': True
}

print("Evidence Bundle:")
print(json.dumps(evidence, indent=2))
print()

# ─── Write to S3 ────────────────────────────────────────────
evidence_key  = f'{prefix}/release/evidence.json'
evidence_body = json.dumps(evidence, indent=2).encode()

if RUN_AWS:
    s3.put_object(Bucket=bucket, Key=evidence_key, Body=evidence_body)
    print(f"✅ Evidence written to: s3://{bucket}/{evidence_key}")
else:
    MockAWS.put_object(bucket, evidence_key, evidence_body)

print()
print("🔍 In a real MLOps platform, this evidence would be:")
print("   → Attached to the model version in the registry")
print("   → Reviewed by a model approver before deployment")
print("   → Checked by CI/CD before auto-deployment gates")


Evidence Bundle:
{
  "artifact_uri": "s3://bits-ai-platform-engineering-111122223333-us-east-2/session_01/runs/20260910070242/",
  "checks": [
    "schema or contract reviewed",
    "logs available",
    "failure path defined",
    "rollback or recovery path documented",
    "cost cleanup listed"
  ],
  "service_owners": {
    "technical": "instructor",
    "business": "programme owner"
  },
  "created_at_utc": "2026-09-10T07:02:42.618131",
  "run_id": "86634a94-ad53-473d-9c43-3d408143963e",
  "cleanup_required": true
}

  📦 S3 PUT  → s3://bits-ai-platform-engineering-111122223333-us-east-2/session_01/runs/20260910070242/release/evidence.json
     Size   : 508 bytes

🔍 In a real MLOps platform, this evidence would be:
   → Attached to the model version in the registry
   → Reviewed by a model approver before deployment
   → Checked by CI/CD before auto-deployment gates


/tmp/ipykernel_2337/2923671954.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at_utc' : datetime.datetime.utcnow().isoformat(),


In [23]:
# ============================================================
# 🧹 CELL: Implementation B — Cleanup Checklist
# ============================================================
#
# WHY CLEANUP MATTERS:
#
# AWS charges for running resources. Forgotten resources cost real money.
# Common "cloud bill horror stories":
#   → A developer left a GPU endpoint running for 3 months: $8,000 bill
#   → An EKS cluster forgotten over a long weekend: $1,200 bill
#   → 100 TB of undeleted S3 snapshots: $2,300/month ongoing
#
# Rule: if you created it, you clean it up. No exceptions.

cleanup_items = [
    {
        'resource': 'SageMaker Endpoints',
        'action'  : 'Delete temporary endpoints or endpoint configs',
        'why'     : 'Endpoints charge per hour even when idle ($0.20–$5/hr)',
        'done'    : False
    },
    {
        'resource': 'Orchestration Resources',
        'action'  : 'Stop/remove EKS, ECS, MWAA, Step Functions resources',
        'why'     : 'Managed environments have always-on charges',
        'done'    : False
    },
    {
        'resource': 'EventBridge Schedules',
        'action'  : 'Disable temporary EventBridge schedules and rules',
        'why'     : 'Triggers can fire indefinitely and accumulate charges',
        'done'    : False
    },
    {
        'resource': 'CloudWatch Logs',
        'action'  : 'Confirm log retention policy (set to 7–30 days)',
        'why'     : 'Unlimited retention = unlimited storage costs',
        'done'    : False
    },
    {
        'resource': 'S3 Evidence',
        'action'  : 'Keep S3 evidence ONLY if required by course runbook',
        'why'     : 'S3 is cheap but bulk data adds up over time',
        'done'    : False
    },
]

print("🧹 Cleanup Checklist (mark 'done': True when complete):")
print()
for i, item in enumerate(cleanup_items, 1):
    status = '✅' if item['done'] else '[ ]'
    print(f"  {status} {i}. {item['resource']}")
    print(f"       Action: {item['action']}")
    print(f"       Why   : {item['why']}")
    print()

# ─── Write checklist to S3 ──────────────────────────────────
checklist_key  = f'{prefix}/cleanup/checklist.json'
checklist_body = json.dumps(cleanup_items, indent=2).encode()

if RUN_AWS:
    s3.put_object(Bucket=bucket, Key=checklist_key, Body=checklist_body)
    print(f"✅ Checklist written to: s3://{bucket}/{checklist_key}")
else:
    MockAWS.put_object(bucket, checklist_key, checklist_body)

print()
print("📌 Before ending any lab session, run through this list.")
print("   Instructors WILL check AWS cost dashboards after sessions!")


🧹 Cleanup Checklist (mark 'done': True when complete):

  [ ] 1. SageMaker Endpoints
       Action: Delete temporary endpoints or endpoint configs
       Why   : Endpoints charge per hour even when idle ($0.20–$5/hr)

  [ ] 2. Orchestration Resources
       Action: Stop/remove EKS, ECS, MWAA, Step Functions resources
       Why   : Managed environments have always-on charges

  [ ] 3. EventBridge Schedules
       Action: Disable temporary EventBridge schedules and rules
       Why   : Triggers can fire indefinitely and accumulate charges

  [ ] 4. CloudWatch Logs
       Action: Confirm log retention policy (set to 7–30 days)
       Why   : Unlimited retention = unlimited storage costs

  [ ] 5. S3 Evidence
       Action: Keep S3 evidence ONLY if required by course runbook
       Why   : S3 is cheap but bulk data adds up over time

  📦 S3 PUT  → s3://bits-ai-platform-engineering-111122223333-us-east-2/session_01/runs/20260910070242/cleanup/checklist.json
     Size   : 984 bytes

📌 Befor

---
## 📌 SECTION 5 (continued) — Stakeholder Perspectives

### 👥 Who Cares About Your ML Model (and What They Want)

One of the most overlooked aspects of MLOps: **you're not just shipping code, you're working with people**.

---

**🔧 DevOps / Infrastructure Team**  
*"We want ML to work like any other software service."*

What they need from you:
- A Docker image they can deploy (not "it runs in my Jupyter notebook")
- An API endpoint with documented input/output schemas
- Health check endpoints (`/health`, `/metrics`)
- Resource requirements (CPU/RAM/GPU) in writing

---

**🧪 QA / Testing Team**  
*"We need to be able to test your model like we test any feature."*

What they need from you:
- Test datasets with known expected outputs
- A documented acceptance threshold (e.g., "F1 > 0.85 on holdout set")
- Regression test suite (does the new model break existing use cases?)
- A staging environment where they can test before production

---

**🔒 Security / Compliance Team**  
*"We're responsible if this model causes harm or leaks data."*

What they need from you:
- PII audit: does the model process personal data? Is it encrypted?
- Model card: training data provenance, known biases, intended use
- Access logs: who can query the model?
- Incident response plan: if the model produces harmful outputs, how do we shut it down in < 10 minutes?

---

**📊 Product Team**  
*"We want the model to drive business outcomes, not just have good metrics."*

What they need from you:
- Business metric tracking (not just accuracy — how does it affect revenue/NPS?)
- A/B test results before full rollout
- Interpretability: "why did the model predict X for this customer?"
- Escalation path for edge cases humans should handle

---

> 🎯 **Key insight:** MLOps practitioners are **translators**. They speak data science to engineers, speak engineering to product, and speak both to security.


In [24]:
# ============================================================
# 📊 CELL: Generate a Stakeholder-Ready Production Report
# ============================================================
#
# This simulates what a mature MLOps system would auto-generate
# at the end of a model deployment pipeline.
#
# Try it: Change any of the values below and re-run the cell!

model_metrics = {
    'accuracy'       : 0.923,
    'precision'      : 0.911,
    'recall'         : 0.936,
    'f1_score'       : 0.923,
    'auc_roc'        : 0.971,
    'latency_p50_ms' : 45,
    'latency_p99_ms' : 120,
}

deployment_config = {
    'model_name'     : 'churn-predictor-v3',
    'model_version'  : '3.1.2',
    'instance_type'  : 'ml.m5.large',
    'min_instances'  : 1,
    'max_instances'  : 10,    # ← auto-scale up to 10
    'traffic_pct'    : 10,    # ← start at 10% traffic (canary deploy)
}

# ─── Stakeholder report ─────────────────────────────────────
print("=" * 65)
print("  MODEL DEPLOYMENT REPORT")
print(f"  {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
print("=" * 65)

print("\n📊 FOR: DATA SCIENCE TEAM")
print("─" * 40)
for k, v in model_metrics.items():
    bar = "█" * int((v if v <= 1 else v/200) * 20)
    print(f"  {k:<18}: {v}")

print("\n⚙️  FOR: DEVOPS / INFRA TEAM")
print("─" * 40)
for k, v in deployment_config.items():
    print(f"  {k:<18}: {v}")
print(f"  {'endpoint_name':<18}: {deployment_config['model_name']}-endpoint")

# ─── Gate evaluation ────────────────────────────────────────
print("\n🚦 FOR: QA / RELEASE TEAM — Gate Status")
print("─" * 40)
gates = {
    'Accuracy > 0.90'    : model_metrics['accuracy'] > 0.90,
    'F1 > 0.90'          : model_metrics['f1_score'] > 0.90,
    'P99 latency < 200ms': model_metrics['latency_p99_ms'] < 200,
    'Traffic canary ≤ 20%': deployment_config['traffic_pct'] <= 20,
}
all_pass = all(gates.values())
for gate, passed in gates.items():
    print(f"  {'✅' if passed else '❌'} {gate}")
print()
print(f"  Overall: {'✅ APPROVED FOR DEPLOYMENT' if all_pass else '❌ BLOCKED — fix failing gates'}")

print("\n🔒 FOR: SECURITY TEAM")
print("─" * 40)
print("  ✅ IAM role: SageMakerExecutionRole (least-privilege)")
print("  ✅ S3 encryption: AES-256 (SSE-S3)")
print("  ✅ No PII in training data (validated by DLP scan)")
print("  ✅ CloudTrail logging: enabled")
print("  ✅ VPC deployment: yes (no public internet access)")


  MODEL DEPLOYMENT REPORT
  2026-09-10 07:02 UTC

📊 FOR: DATA SCIENCE TEAM
────────────────────────────────────────
  accuracy          : 0.923
  precision         : 0.911
  recall            : 0.936
  f1_score          : 0.923
  auc_roc           : 0.971
  latency_p50_ms    : 45
  latency_p99_ms    : 120

⚙️  FOR: DEVOPS / INFRA TEAM
────────────────────────────────────────
  model_name        : churn-predictor-v3
  model_version     : 3.1.2
  instance_type     : ml.m5.large
  min_instances     : 1
  max_instances     : 10
  traffic_pct       : 10
  endpoint_name     : churn-predictor-v3-endpoint

🚦 FOR: QA / RELEASE TEAM — Gate Status
────────────────────────────────────────
  ✅ Accuracy > 0.90
  ✅ F1 > 0.90
  ✅ P99 latency < 200ms
  ✅ Traffic canary ≤ 20%

  Overall: ✅ APPROVED FOR DEPLOYMENT

🔒 FOR: SECURITY TEAM
────────────────────────────────────────
  ✅ IAM role: SageMakerExecutionRole (least-privilege)
  ✅ S3 encryption: AES-256 (SSE-S3)
  ✅ No PII in training data (validated 

/tmp/ipykernel_2337/2174992359.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"  {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")


---
## 📌 SECTION 6 — Practice Zone: Try It Yourself (No AWS Required!)

### 🛠️ Local Simulation: Mini MLOps Pipeline

This section gives you a **fully local MLOps mini-pipeline** to experiment with.  
No AWS account. No costs. No waiting for cloud resources.

**What this simulates:**
1. Data validation (checks for nulls, schema mismatches, drift)
2. Model training (a simple classifier on toy data)
3. Model evaluation (accuracy gate)
4. Registry registration (a local JSON "registry")
5. Evidence bundle generation

**Try modifying:**
- The `ACCURACY_GATE` threshold
- The `inject_drift` flag
- The `schema` fields

This builds intuition for what SageMaker does automatically in production.


In [25]:
ACCURACY_GATE  = 0.85    # Model must beat this to pass (try 0.95 to see it fail)
inject_drift   = False   # Set True to simulate data drift and trigger a block
feature_schema = ['age', 'income', 'tenure', 'product_code']   # Expected columns

# ─── Simulate data validation ───────────────────────────────
import random
random.seed(42)

# Generate fake data (simulates a new batch arriving for retraining)
if inject_drift:
    # Drift: 'tenure' column has been removed (schema mismatch!)
    incoming_features = ['age', 'income', 'product_code']
else:
    incoming_features = feature_schema.copy()

print("=" * 60)
print("STEP 1: DATA VALIDATION")
print("─" * 60)
missing = set(feature_schema) - set(incoming_features)
extra   = set(incoming_features) - set(feature_schema)
drift_detected = bool(missing or extra)

print(f"  Expected schema  : {feature_schema}")
print(f"  Incoming schema  : {incoming_features}")
print(f"  Missing columns  : {list(missing) if missing else 'None ✅'}")
print(f"  Extra columns    : {list(extra) if extra else 'None ✅'}")
print(f"  Schema drift?    : {'⚠️  YES — pipeline blocked!' if drift_detected else 'No ✅'}")

if drift_detected:
    print()
    print("  🚨 Pipeline HALTED at data validation stage.")
    print("     Alert sent to: data-engineering-team@bits-pilani.ac.in")
    print("     Next step: investigate schema change and update pipeline.")
else:
    # ─── Simulate model training ────────────────────────────
    print()
    print("=" * 60)
    print("STEP 2: MODEL TRAINING (sklearn LogisticRegression)")
    print("─" * 60)

    try:
        from sklearn.datasets import make_classification
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import accuracy_score, f1_score

        X, y = make_classification(n_samples=1000, n_features=4,
                                   n_informative=2, random_state=42)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train, y_train)

        y_pred    = model.predict(X_test)
        accuracy  = accuracy_score(y_test, y_pred)
        f1        = f1_score(y_test, y_pred)

        print(f"  Training samples : {len(X_train)}")
        print(f"  Test samples     : {len(X_test)}")
        print(f"  Accuracy         : {accuracy:.4f}")
        print(f"  F1 Score         : {f1:.4f}")

    except ImportError:
        # Fallback if sklearn not installed
        accuracy = round(random.uniform(0.83, 0.95), 4)
        f1       = round(accuracy - 0.02, 4)
        print(f"  [sklearn not installed — using simulated metrics]")
        print(f"  Accuracy : {accuracy}")
        print(f"  F1 Score : {f1}")

    # ─── Accuracy gate ──────────────────────────────────────
    print()
    print("=" * 60)
    print("STEP 3: EVALUATION GATE")
    print("─" * 60)
    gate_pass = accuracy >= ACCURACY_GATE
    print(f"  Accuracy gate threshold : {ACCURACY_GATE}")
    print(f"  Model accuracy          : {accuracy:.4f}")
    print(f"  Gate result             : {'✅ PASSED' if gate_pass else '❌ FAILED'}")

    if gate_pass:
        # ─── Registry registration ──────────────────────────
        print()
        print("=" * 60)
        print("STEP 4: MODEL REGISTRY REGISTRATION")
        print("─" * 60)
        model_record = {
            'model_name'    : 'churn-predictor',
            'version'       : '1.0.0',
            'accuracy'      : round(accuracy, 4),
            'f1_score'      : round(f1, 4),
            'gate_threshold': ACCURACY_GATE,
            'status'        : 'Approved',
            'registered_at' : datetime.datetime.utcnow().isoformat(),
            'approved_by'   : 'auto-gate',
        }
        print(f"  Model registered to local registry:")
        print(json.dumps(model_record, indent=4))

        # ─── Evidence bundle ────────────────────────────────
        print()
        print("=" * 60)
        print("STEP 5: EVIDENCE BUNDLE")
        print("─" * 60)
        print(f"  ✅ Data validation passed")
        print(f"  ✅ Model trained on {len(X_train)} samples")
        print(f"  ✅ Accuracy gate passed ({accuracy:.4f} ≥ {ACCURACY_GATE})")
        print(f"  ✅ Model registered (version {model_record['version']})")
        print(f"  ⬜ Deployment pending human approval")
        print()
        print("  🎉 Pipeline complete! Model is ready for deployment review.")
    else:
        print()
        print("  🚨 Model BLOCKED — does not meet accuracy threshold.")
        print(f"     Deficit: {ACCURACY_GATE - accuracy:.4f} below gate")
        print("     Action: Tune hyperparameters or add more training data.")

STEP 1: DATA VALIDATION
────────────────────────────────────────────────────────────
  Expected schema  : ['age', 'income', 'tenure', 'product_code']
  Incoming schema  : ['age', 'income', 'tenure', 'product_code']
  Missing columns  : None ✅
  Extra columns    : None ✅
  Schema drift?    : No ✅

STEP 2: MODEL TRAINING (sklearn LogisticRegression)
────────────────────────────────────────────────────────────
  Training samples : 800
  Test samples     : 200
  Accuracy         : 0.8850
  F1 Score         : 0.8808

STEP 3: EVALUATION GATE
────────────────────────────────────────────────────────────
  Accuracy gate threshold : 0.85
  Model accuracy          : 0.8850
  Gate result             : ✅ PASSED

STEP 4: MODEL REGISTRY REGISTRATION
────────────────────────────────────────────────────────────
  Model registered to local registry:
{
    "model_name": "churn-predictor",
    "version": "1.0.0",
    "accuracy": 0.885,
    "f1_score": 0.8808,
    "gate_threshold": 0.85,
    "status": "App

/tmp/ipykernel_2337/1504908427.py:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'registered_at' : datetime.datetime.utcnow().isoformat(),


### 🎯 Challenge Exercises

Try these modifications to the local simulation above:

---

**Exercise 1 — Trigger the Drift Alert**
```python
inject_drift = True   # ← Set this and re-run the cell
```
*Observe:* The pipeline stops at Step 1. This is what a mature MLOps system does automatically — it never trains on bad data.

---

**Exercise 2 — Set an Impossible Gate**
```python
ACCURACY_GATE = 0.99   # ← Near-impossible for logistic regression on this data
```
*Observe:* The model fails the gate at Step 3. Try lowering it to `0.80` to see it pass.

---

**Exercise 3 — Add a New Feature to the Schema**
```python
feature_schema = ['age', 'income', 'tenure', 'product_code', 'city']
```
*Observe:* Since the incoming data doesn't have 'city', drift is detected. This mirrors real incidents where a data engineering team silently changes schema.

---

> 💡 **Reflection question:** What would you add to this pipeline to handle each failure mode automatically?


In [26]:
# ============================================================
# 🏆 FINAL QUIZ — Session 01 Comprehensive Check
# ============================================================

def run_final_quiz(answers):
    correct_answers = {
        1: ('c', 'Silent model degradation — accuracy drops without errors or alerts. '
                  'This is unique to ML systems and is why monitoring is non-negotiable.'),
        2: ('b', 'Feature stores solve training-serving skew by ensuring the same '
                  'feature engineering code runs at training time AND serving time.'),
        3: ('a', 'Tags help you track costs per project, auto-expire test resources, '
                  'and identify resource owners. They are the foundation of cloud governance.'),
        4: ('d', 'Level 3 is characterized by a fully automated CT/CD pipeline where '
                  'new data triggers retraining, models are evaluated, and deploy automatically.'),
        5: ('b', 'The evidence bundle is the formal record that all production gates '
                  'passed, linking the model artifact to its validation history.'),
    }

    questions = {
        1: "What is the most dangerous failure mode unique to ML systems?\n"
           "   a) Server crashes           b) Network timeouts\n"
           "   c) Silent model degradation d) Disk full errors",
        2: "Feature stores primarily solve which problem?\n"
           "   a) Slow model training       b) Training-serving skew\n"
           "   c) Expensive GPU costs       d) Version control for code",
        3: "Why should every AWS resource have 'project', 'owner', and 'expires' tags?\n"
           "   a) For cost tracking and governance\n"
           "   b) AWS requires tags for SageMaker jobs\n"
           "   c) Tags enable CloudWatch monitoring\n"
           "   d) They improve model performance",
        4: "Which MLOps maturity level has a fully automated CT/CD pipeline?\n"
           "   a) Level 0  b) Level 1  c) Level 2  d) Level 3",
        5: "What is the primary purpose of the evidence bundle in our session?\n"
           "   a) To train the model         b) To prove production readiness\n"
           "   c) To configure IAM roles     d) To set up CloudWatch alarms",
    }

    score = 0
    print("=" * 65)
    print("  FINAL QUIZ — Session 01: End-to-End MLOps")
    print("=" * 65)
    print()
    for q_num, question in questions.items():
        print(f"Q{q_num}: {question}")
        your_ans = answers.get(q_num, '?').lower()
        correct  = correct_answers[q_num][0]
        explanation = correct_answers[q_num][1]
        if your_ans == correct:
            score += 1
            print(f"     → ✅ {your_ans.upper()} — Correct!")
        else:
            print(f"     → ❌ Your answer: {your_ans.upper() if your_ans != '?' else '(not set)'}  |  Correct: {correct.upper()}")
        print(f"        💡 {explanation}")
        print()

    pct = (score / len(questions)) * 100
    print("─" * 65)
    print(f"  Score: {score}/{len(questions)}  ({pct:.0f}%)")
    if pct == 100:
        print("  🏆 Perfect! You're ready for Lab Mode.")
    elif pct >= 80:
        print("  ✅ Great work! Review the concepts you missed.")
    elif pct >= 60:
        print("  📖 Good attempt. Re-read sections 1–3 before the live lab.")
    else:
        print("  📚 Take some time to revisit the concept cells above.")
    print("=" * 65)

# ─── YOUR ANSWERS — change a/b/c/d for each question ────────
final_answers = {
    1: "c",   # ← your answer for Q1
    2: "b",   # ← your answer for Q2
    3: "a",   # ← your answer for Q3
    4: "d",   # ← your answer for Q4
    5: "b",   # ← your answer for Q5
}

run_final_quiz(final_answers)


  FINAL QUIZ — Session 01: End-to-End MLOps

Q1: What is the most dangerous failure mode unique to ML systems?
   a) Server crashes           b) Network timeouts
   c) Silent model degradation d) Disk full errors
     → ✅ C — Correct!
        💡 Silent model degradation — accuracy drops without errors or alerts. This is unique to ML systems and is why monitoring is non-negotiable.

Q2: Feature stores primarily solve which problem?
   a) Slow model training       b) Training-serving skew
   c) Expensive GPU costs       d) Version control for code
     → ✅ B — Correct!
        💡 Feature stores solve training-serving skew by ensuring the same feature engineering code runs at training time AND serving time.

Q3: Why should every AWS resource have 'project', 'owner', and 'expires' tags?
   a) For cost tracking and governance
   b) AWS requires tags for SageMaker jobs
   c) Tags enable CloudWatch monitoring
   d) They improve model performance
     → ✅ A — Correct!
        💡 Tags help you tra

---
## 📌 SECTION 7 — Summary & Next Steps

### ✅ What You Covered Today

| Concept | Key Takeaway |
|---------|--------------|
| **Production Gap** | Notebooks are for experimentation; production needs reproducibility, monitoring, and governance |
| **MLOps vs DevOps** | MLOps adds data versioning, model monitoring, and continuous training to the DevOps toolkit |
| **Maturity Levels** | Most teams are at Level 0–1; this course gets you to Level 2–3 |
| **AWS Architecture** | S3 (storage) → SageMaker Processing (compute) → Model Registry (catalog) → CloudWatch (observe) |
| **Manifest** | Write it first, before any resource creation — it's your audit trail |
| **Evidence Bundle** | Links the model artifact to its validation history; gates must pass before deployment |
| **Stakeholder Needs** | DevOps wants APIs, QA wants test data, Security wants audit logs, Product wants business metrics |
| **Cleanup** | Never leave AWS resources running; tag everything with an expiry date |

---

### 🗓️ What's Coming Next

```
Session 02  →  Pipeline Orchestration Basics (Apache Airflow, DAGs)
Session 03  →  Advanced Pipeline Orchestration (Kubeflow, Ray, MLflow)
Session 04  →  Containerization for AI/ML (Docker, CUDA, multi-stage builds)
```

**Before Session 02:**
- [ ] Re-run this notebook in Lab Mode (when sandbox is available)
- [ ] Explore the SageMaker Model Registry in the AWS Console
- [ ] Read about Apache Airflow DAG concepts (links in Week 2 COD)
- [ ] Reflect: What level is your current team/workplace?

---

### 📚 Recommended Reading

- [AWS MLOps Foundation Roadmap](https://aws.amazon.com/blogs/machine-learning/mlops-foundation-roadmap-for-enterprises-with-amazon-sagemaker/)
- [Google's Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml)
- [Martin Fowler — Continuous Delivery for Machine Learning](https://martinfowler.com/articles/cd4ml.html)
- [Chip Huyen — Designing Machine Learning Systems](https://www.oreilly.com/library/view/designing-machine-learning/9781098107956/) (Chapters 1–3)

---

> **Remember:** The goal of MLOps is not to make ML harder — it's to make it **sustainable at scale**. Every practice you learned today is solving a real problem that teams hit when they try to go from one model to one hundred.

---
*BITS Pilani Professional AI/ML Programme · Module 05 · Session 01*


---
## 👨‍🏫 Instructor Reference (not shown to students in presentation mode)

### Teaching Notes

| Segment | Time | Focus |
|---------|------|-------|
| Sections 1–2 | 0:00–0:20 | Walk through concept cells; ask students to predict answers |
| Section 3 | 0:20–0:30 | Whiteboard the AWS architecture; relate each service to a real-world analogy |
| Setup cells | 0:30–0:40 | Verify sandbox access; show IAM role, S3 bucket creation |
| Implementation A | 0:40–1:10 | Live coding; run with RUN_AWS=True; highlight the manifest first |
| Checkpoint | 1:10–1:20 | Run Knowledge Check 1; discuss Q2 (S3 naming) — common interview question |
| Implementation B | 1:20–1:50 | Live coding; run release gates; have students change thresholds |
| Practice Zone | 1:50–2:10 | Students work through exercises independently; instructor circulates |
| Final Quiz | 2:10–2:20 | Students complete; debrief Q1 (silent degradation) and Q2 (feature stores) |
| Cleanup | 2:20–2:30 | Run cleanup checklist live; show CloudWatch logs; end on handoff |

### Common Student Questions

**Q: "Can I use MLflow instead of SageMaker Model Registry?"**  
A: Yes — the concepts are the same. MLflow is framework-agnostic; SageMaker Registry is AWS-native with tighter integration to SageMaker Pipelines. We use SageMaker here because the course is AWS-focused.

**Q: "Does the manifest have to be JSON?"**  
A: No — YAML is common too. JSON is easier to parse programmatically. What matters is the structure and that it's written to a queryable location.

**Q: "What's the difference between a model artifact and a model package?"**  
A: The artifact is the file (model.pkl, model.tar.gz). The package is the artifact + metadata (training metrics, inference image, deployment config). The registry stores packages, not raw artifacts.

### Sandbox Verification Checklist (before session)
- [ ] Sandbox account has SageMakerExecutionRole configured
- [ ] S3 bucket creation permissions verified
- [ ] CloudWatch log group exists or permissions to create it
- [ ] Processing job container image is available (ECR or public)
- [ ] Estimated session cost confirmed (< $5 for this session)


In [27]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-10 12:32:42
